# On-device ONNX pipeline validation

This notebook reproduces the MoveYourBody semantic matching pipeline in Python using the **same files the app ships with**, so results here line up with what runs on the device.

| Data source | Used by the app in | Used here for |
|---|---|---|
| `assets/models/model_quantized.onnx` | `OnnxService` | Running the model |
| `assets/models/vocab.txt` | `TokenizerService` (Dart WordPiece) | Reference + Dart-equivalent tokenization |
| `assets/exercises/exercises.json` | `TagSetupRepository.processAndSeedTags` | Extracting goal / injury tags |

It covers:
1. Extracting tags exactly like `TagSetupRepository`
2. Tokenizer parity between the Dart WordPiece tokenizer and the reference BERT tokenizer
3. The effect of tokenization differences on embeddings
4. CLS vs mean pooling
5. Similarity threshold calibration for tag matching
6. Exporting `test/fixtures/tokenizer_parity.json`, which `test/tokenizer_parity_test.dart` checks against the real Dart tokenizer

Run from the `notebooks/` directory after `pip install -r requirements.txt`.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import onnxruntime as ort
from tokenizers import BertWordPieceTokenizer, Tokenizer, models, pre_tokenizers

REPO = Path.cwd().parent
MODEL_PATH = REPO / "assets/models/model_quantized.onnx"
VOCAB_PATH = REPO / "assets/models/vocab.txt"
EXERCISES_PATH = REPO / "assets/exercises/exercises.json"
FIXTURE_PATH = REPO / "test/fixtures/tokenizer_parity.json"

for p in (MODEL_PATH, VOCAB_PATH, EXERCISES_PATH):
    print(f"{p.relative_to(REPO).as_posix()}: {p.stat().st_size / 1024:.0f} KB")

assets/models/model_quantized.onnx: 22308 KB
assets/models/vocab.txt: 226 KB
assets/exercises/exercises.json: 124 KB


## 1. Tags from `exercises.json`

Mirrors `TagSetupRepository.processAndSeedTags`: `goal_tags` and `contraindications` are `;`-separated and trimmed.

In [2]:
exercises = json.loads(EXERCISES_PATH.read_text(encoding="utf-8"))["exercises"]

def split_tags(field):
    return {t.strip() for ex in exercises for t in (ex.get(field) or "").split(";") if t.strip()}

goal_tags = sorted(split_tags("goal_tags"))
injury_tags = sorted(split_tags("contraindications"))
all_tags = goal_tags + injury_tags

print(f"{len(exercises)} exercises -> {len(goal_tags)} goal tags, {len(injury_tags)} injury tags")
print("multi-word (snake_case) tags:", sum("_" in t for t in all_tags))
print("examples:", all_tags[:8])

89 exercises -> 44 goal tags, 89 injury tags
multi-word (snake_case) tags: 92
examples: ['abdominal', 'athletic_performance', 'back', 'back_pain_relief', 'balance', 'bodyweight', 'breathing', 'cardio']


## 2. Model inputs and outputs

In [3]:
session = ort.InferenceSession(str(MODEL_PATH))
for i in session.get_inputs():
    print("input ", i.name, i.shape, i.type)
for o in session.get_outputs():
    print("output", o.name, o.shape, o.type)

input  input_ids ['batch_size', 'sequence_length'] tensor(int64)
input  attention_mask ['batch_size', 'sequence_length'] tensor(int64)
input  token_type_ids ['batch_size', 'sequence_length'] tensor(int64)
output last_hidden_state ['batch_size', 'sequence_length', 384] tensor(float)


## 3. Tokenizers

* **Reference**: the BERT uncased WordPiece tokenizer the model was trained with (lowercase, accent stripping, punctuation split into separate tokens).
* **Dart-equivalent**: a Python port of `dart_wordpiece` with `normalizeText: true`, as configured in `TokenizerService`. It lowercases, replaces characters matching `[^\p{L}\p{N}_\s]` with spaces, splits on whitespace and runs greedy WordPiece. **Underscores are kept as part of the word.**

The port was checked against ids dumped from the real Dart tokenizer for every tag (with and without underscores) plus sample queries, and matched on all of them except text with accented letters, where Dart emits `[UNK]`.

In [4]:
reference_tok = BertWordPieceTokenizer(str(VOCAB_PATH), lowercase=True)
vocab = {line: i for i, line in enumerate(VOCAB_PATH.read_text(encoding="utf-8").splitlines())}
inv_vocab = {i: t for t, i in vocab.items()}
CLS, SEP = vocab["[CLS]"], vocab["[SEP]"]

_wordpiece = Tokenizer(models.WordPiece(vocab, unk_token="[UNK]", max_input_chars_per_word=100))
_wordpiece.pre_tokenizer = pre_tokenizers.WhitespaceSplit()

def dart_normalize(text):
    lower = text.strip().lower()
    cleaned = re.sub(r"[^\w\s]", " ", lower)  # \w keeps letters, digits and "_"
    words = [w for w in re.split(r"\s+", cleaned) if w]
    return " ".join(words) or lower

def dart_ids(text):
    return [CLS] + _wordpiece.encode(dart_normalize(text)).ids + [SEP]

def reference_ids(text):
    return reference_tok.encode(text).ids

def show(ids):
    return [inv_vocab[i] for i in ids]

for t in ["ankle_injury", "ankle injury", "My knee hurts!"]:
    print(f"{t!r}\n  dart      {show(dart_ids(t))}\n  reference {show(reference_ids(t))}")

'ankle_injury'
  dart      ['[CLS]', 'ankle', '##_', '##in', '##ju', '##ry', '[SEP]']
  reference ['[CLS]', 'ankle', '_', 'injury', '[SEP]']
'ankle injury'
  dart      ['[CLS]', 'ankle', 'injury', '[SEP]']
  reference ['[CLS]', 'ankle', 'injury', '[SEP]']
'My knee hurts!'
  dart      ['[CLS]', 'my', 'knee', 'hurts', '[SEP]']
  reference ['[CLS]', 'my', 'knee', 'hurts', '!', '[SEP]']


## 4. Tokenizer parity on app tags

Compare token ids for every tag as stored (`snake_case`) and with underscores replaced by spaces.

In [5]:
def parity(texts):
    return sum(dart_ids(t) == reference_ids(t) for t in texts)

as_stored = all_tags
as_text = [t.replace("_", " ") for t in all_tags]

print(f"tags as stored     : {parity(as_stored)}/{len(as_stored)} match")
print(f"underscores->spaces: {parity(as_text)}/{len(as_text)} match")

for t in [t for t in as_stored if dart_ids(t) != reference_ids(t)][:3]:
    print(f"\n{t}\n  dart      {show(dart_ids(t))}\n  reference {show(reference_ids(t))}")

tags as stored     : 41/133 match
underscores->spaces: 133/133 match

athletic_performance
  dart      ['[CLS]', 'athletic', '##_', '##per', '##form', '##ance', '[SEP]']
  reference ['[CLS]', 'athletic', '_', 'performance', '[SEP]']

back_pain_relief
  dart      ['[CLS]', 'back', '##_', '##pa', '##in', '##_', '##rel', '##ie', '##f', '[SEP]']
  reference ['[CLS]', 'back', '_', 'pain', '_', 'relief', '[SEP]']

chest_opening
  dart      ['[CLS]', 'chest', '##_', '##open', '##ing', '[SEP]']
  reference ['[CLS]', 'chest', '_', 'opening', '[SEP]']


Every multi-word tag breaks on the Dart side: `_` is glued to the following word, so `injury` becomes `##in ##ju ##ry`. Replacing `_` with a space before tokenizing gives exact parity for all tags.

## 5. Impact on embeddings

Embed each multi-word tag with the reference tokenization (what the model expects) and with Dart's tokenization, and measure how close the two vectors are. Embeddings use mean pooling + L2 normalization (section 6).

In [6]:
def embed_ids(ids, pooling="mean"):
    ids = np.array([ids], dtype=np.int64)
    feed = {"input_ids": ids, "attention_mask": np.ones_like(ids), "token_type_ids": np.zeros_like(ids)}
    hidden = session.run(None, feed)[0][0]
    v = hidden[0] if pooling == "cls" else hidden.mean(axis=0)
    return v / np.linalg.norm(v)

def agreement(texts):
    sims = np.array([embed_ids(dart_ids(t)) @ embed_ids(reference_ids(t)) for t in texts])
    return sims.mean(), sims.min()

multi = [t for t in all_tags if "_" in t]
for label, texts in [("as stored", multi), ("underscores->spaces", [t.replace("_", " ") for t in multi])]:
    avg, lo = agreement(texts)
    print(f"{label:20} cosine(dart, reference): mean {avg:.3f}, min {lo:.3f}")

as stored            cosine(dart, reference): mean 0.630, min 0.299


underscores->spaces  cosine(dart, reference): mean 1.000, min 1.000


## 6. CLS vs mean pooling

`all-MiniLM-L6-v2` is a sentence-transformers model trained with **mean pooling** over real tokens followed by L2 normalization. Taking the `[CLS]` vector instead makes everything look similar. Below: how many tags score at least 0.60 for sample queries.

In [7]:
def embed(text, pooling="mean"):
    return embed_ids(reference_ids(text), pooling)

QUERIES = [
    ("goal", "I want to lose belly fat"), ("goal", "build stronger legs"),
    ("goal", "improve flexibility"), ("goal", "reduce stress"),
    ("goal", "get fit and healthy"), ("goal", "better sleep"),
    ("injury", "my knee hurts"), ("injury", "lower back pain"),
    ("injury", "shoulder injury"), ("injury", "high blood pressure"),
    ("injury", "I had knee surgery"), ("injury", "sprained ankle"),
]
TAGS = {"goal": goal_tags, "injury": injury_tags}

tag_vecs = {
    pooling: {t: embed(t.replace("_", " "), pooling) for t in all_tags}
    for pooling in ("cls", "mean")
}

def scores(kind, query, pooling):
    q = embed(query, pooling)
    return sorted(((float(q @ tag_vecs[pooling][t]), t) for t in TAGS[kind]), reverse=True)

print(f"{'query':28} {'CLS >=0.60':>12} {'mean >=0.60':>12}")
for kind, q in QUERIES:
    c = sum(s >= 0.6 for s, _ in scores(kind, q, "cls"))
    m = sum(s >= 0.6 for s, _ in scores(kind, q, "mean"))
    n = len(TAGS[kind])
    print(f"{q:28} {f'{c}/{n}':>12} {f'{m}/{n}':>12}")

query                          CLS >=0.60  mean >=0.60
I want to lose belly fat            35/44         0/44
build stronger legs                 43/44         0/44
improve flexibility                 44/44         1/44
reduce stress                       44/44         1/44
get fit and healthy                 44/44         0/44
better sleep                        44/44         1/44
my knee hurts                       88/89         5/89
lower back pain                     88/89         3/89
shoulder injury                     87/89        14/89
high blood pressure                 84/89         2/89
I had knee surgery                  88/89         4/89
sprained ankle                      85/89         4/89


With CLS pooling almost every tag passes the threshold, so matching can't discriminate. Mean pooling separates relevant tags clearly.

## 7. Threshold calibration (mean pooling)

`TagSetupRepository.findTopMatches` keeps tags with `score >= threshold`, then the top `limit` (3 in the loading screen).

In [8]:
THRESHOLDS = [0.40, 0.45, 0.50, 0.60]
print(f"{'query':28} " + " ".join(f"{t:>5}" for t in THRESHOLDS) + "   top 3")
for kind, q in QUERIES:
    s = scores(kind, q, "mean")
    counts = " ".join(f"{sum(x >= t for x, _ in s):>5}" for t in THRESHOLDS)
    top = ", ".join(f"{t} {x:.2f}" for x, t in s[:3])
    print(f"{q:28} {counts}   {top}")

no_match = {t: sum(scores(k, q, "mean")[0][0] < t for k, q in QUERIES) for t in THRESHOLDS}
print("\nqueries with no match at threshold:", no_match)

query                          0.4  0.45   0.5   0.6   top 3
I want to lose belly fat         2     1     0     0   abdominal 0.46, fat_burn 0.43, cardio 0.30
build stronger legs              5     3     3     0   lower_body_strength 0.54, muscle_building 0.53, strength 0.50
improve flexibility              5     3     2     1   flexibility 0.87, shoulder_stability 0.50, posture 0.49
reduce stress                    2     1     1     1   stress_relief 0.80, relaxation 0.42, lower_body_strength 0.32
get fit and healthy              1     1     0     0   bodyweight 0.46, cardio 0.39, fat_burn 0.39
better sleep                     3     2     1     1   sleep 0.78, warm_up 0.49, warmup 0.41
my knee hurts                   27    16    11     5   knee_pain 0.83, knee_injury 0.75, knee_replacement 0.68
lower back pain                 17    10     7     3   lower_back_pain 1.00, acute_lower_back_pain 0.89, lower_back_injury 0.73
shoulder injury                 36    31    28    14   shoulder_i

At 0.60 several valid goal queries get no match at all. Around 0.45 every sample query keeps its best tag, unrelated tags (typically below 0.40) are dropped, and `limit: 3` caps the rest.

## 8. Export tokenizer parity fixture

Writes reference token ids for every tag (with underscores replaced by spaces, the form `TokenizerService` feeds the tokenizer) plus the sample queries above, which use no punctuation or accents. `test/tokenizer_parity_test.dart` asserts the Dart tokenizer produces exactly these ids.

In [9]:
cases = [{"source": "tag", "input": t} for t in all_tags]
cases += [{"source": "query", "input": q} for _, q in QUERIES]
for c in cases:
    c["ids"] = reference_ids(c["input"].replace("_", " "))

fixture = {
    "generated_by": "notebooks/onnx_pipeline_validation.ipynb",
    "vocab": "assets/models/vocab.txt",
    "cases": cases,
}
FIXTURE_PATH.parent.mkdir(parents=True, exist_ok=True)
FIXTURE_PATH.write_text(json.dumps(fixture, indent=1) + "\n", encoding="utf-8")
print(f"wrote {len(cases)} cases to {FIXTURE_PATH.relative_to(REPO).as_posix()}")
matches = sum(dart_ids(c["input"].replace("_", " ")) == c["ids"] for c in cases)
print(f"dart-equivalent parity on fixture: {matches}/{len(cases)}")

wrote 145 cases to test/fixtures/tokenizer_parity.json
dart-equivalent parity on fixture: 145/145


## Summary

* **Tokenization:** tags must be tokenized as natural text (`_` -> space). As stored in `snake_case`, every multi-word tag is split into sub-word noise by the Dart tokenizer. `TokenizerService` now applies this mapping before encoding.
* **Free text:** Dart drops punctuation instead of emitting punctuation tokens, and turns accented letters into `[UNK]`. Both are documented in `docs/ON_DEVICE_AI.md`.
* **Pooling:** use mean pooling + L2 normalization, not `[CLS]`.
* **Threshold:** with mean pooling, ~0.45 keeps the best match for every sample query; 0.60 drops valid goal matches.